In [0]:
storage_account_name = "segrid"
storage_account_key = "<STORAGE_ACCOUNT_KEY>"  # set via Databricks secret scope 

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
raw_path = f"abfss://raw@{storage_account_name}.dfs.core.windows.net/"
display(dbutils.fs.ls(raw_path))

path,name,size,modificationTime
abfss://raw@segrid.dfs.core.windows.net/household_power_consumption.txt,household_power_consumption.txt,132960755,1784179667000


In [0]:
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Time", StringType(), True),
    StructField("Global_active_power", StringType(), True),
    StructField("Global_reactive_power", StringType(), True),
    StructField("Voltage", StringType(), True),
    StructField("Global_intensity", StringType(), True),
    StructField("Sub_metering_1", StringType(), True),
    StructField("Sub_metering_2", StringType(), True),
    StructField("Sub_metering_3", StringType(), True),
])

df_raw = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .schema(schema)
    .csv(raw_path)
)

print(f"Row count: {df_raw.count()}")
df_raw.printSchema()
display(df_raw.limit(10))

Row count: 2075259
root
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Global_active_power: string (nullable = true)
 |-- Global_reactive_power: string (nullable = true)
 |-- Voltage: string (nullable = true)
 |-- Global_intensity: string (nullable = true)
 |-- Sub_metering_1: string (nullable = true)
 |-- Sub_metering_2: string (nullable = true)
 |-- Sub_metering_3: string (nullable = true)



Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.000
16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.000
16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.000
16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.000
16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.000
16/12/2006,17:29:00,3.520,0.522,235.020,15.000,0.000,2.000,17.000
16/12/2006,17:30:00,3.702,0.520,235.090,15.800,0.000,1.000,17.000
16/12/2006,17:31:00,3.700,0.520,235.220,15.800,0.000,1.000,17.000
16/12/2006,17:32:00,3.668,0.510,233.990,15.800,0.000,1.000,17.000
16/12/2006,17:33:00,3.662,0.510,233.860,15.800,0.000,2.000,16.000


In [0]:
processed_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/energy_readings/"

(
    df_extracted
    .write
    .mode("overwrite")
    .parquet(processed_path)
)

print("Written to processed zone.")
display(dbutils.fs.ls(processed_path))

Written to processed zone.


path,name,size,modificationTime
abfss://processed@segrid.dfs.core.windows.net/energy_readings/_SUCCESS,_SUCCESS,0,1784184041000
abfss://processed@segrid.dfs.core.windows.net/energy_readings/_committed_6721401701983881366,_committed_6721401701983881366,418,1784184041000
abfss://processed@segrid.dfs.core.windows.net/energy_readings/_started_6721401701983881366,_started_6721401701983881366,0,1784184035000
abfss://processed@segrid.dfs.core.windows.net/energy_readings/part-00000-tid-6721401701983881366-a2003c23-644d-4aff-a11d-4fdd055ac2d2-8-1.c000.snappy.parquet,part-00000-tid-6721401701983881366-a2003c23-644d-4aff-a11d-4fdd055ac2d2-8-1.c000.snappy.parquet,5518015,1784184040000
abfss://processed@segrid.dfs.core.windows.net/energy_readings/part-00001-tid-6721401701983881366-a2003c23-644d-4aff-a11d-4fdd055ac2d2-9-1.c000.snappy.parquet,part-00001-tid-6721401701983881366-a2003c23-644d-4aff-a11d-4fdd055ac2d2-9-1.c000.snappy.parquet,5562400,1784184040000
abfss://processed@segrid.dfs.core.windows.net/energy_readings/part-00002-tid-6721401701983881366-a2003c23-644d-4aff-a11d-4fdd055ac2d2-10-1.c000.snappy.parquet,part-00002-tid-6721401701983881366-a2003c23-644d-4aff-a11d-4fdd055ac2d2-10-1.c000.snappy.parquet,5586720,1784184040000
abfss://processed@segrid.dfs.core.windows.net/energy_readings/part-00003-tid-6721401701983881366-a2003c23-644d-4aff-a11d-4fdd055ac2d2-11-1.c000.snappy.parquet,part-00003-tid-6721401701983881366-a2003c23-644d-4aff-a11d-4fdd055ac2d2-11-1.c000.snappy.parquet,5454869,1784184040000
